# Supply-Chain Anomaly Analyst — QLoRA fine-tuning & rigorous evaluation

Fine-tune **Phi-3-mini-4k-instruct** with QLoRA (4-bit NF4) to convert a supply-chain
event feed (JSON) into a structured anomaly assessment (JSON), then **prove** the
improvement against the same base model with a system prompt only:

1. chat-template verification (with an automatic fallback model),
2. **base-model baseline generated FIRST** on the 15 held-out prompts,
3. QLoRA training (assistant-token loss, per-epoch validation loss),
4. merge + push to the Hugging Face Hub,
5. ROUGE-L + BERTScore + **blinded LLM-as-judge** + programmatic checks + manual audit.

Run top to bottom on a **free Colab T4**. Keys come from Colab Secrets; nothing is
hardcoded. The committed dataset and caches make every step reproducible.

In [ ]:
# 1. Setup — repo, dependencies, secrets (never printed)
import os

REPO_URL = "https://github.com/<account>/<repo>.git"
IN_COLAB = "google.colab" in (str(get_ipython()) if "get_ipython" in globals() else "")

if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q -r requirements-colab.txt")
    import subprocess
    if not os.path.exists("/content/repo/.git"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo/supply_chain_finetuning")
else:
    print("not on Colab - assuming the repository checkout is already active")


def _secret(name):
    """Colab Secrets first, then environment, then a typed prompt (never echoed)."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get(name)
    except Exception:
        import getpass

        return getpass.getpass(f"{name}: ")


for _name in ("TEACHER_API_KEY", "JUDGE_API_KEY"):
    os.environ[_name] = _secret(_name)
_hf = os.environ.get("HF_TOKEN", "")
if not _hf:
    try:
        from google.colab import userdata  # type: ignore

        _hf = userdata.get("HF_TOKEN")
    except Exception:
        _hf = ""
if _hf:
    os.environ["HF_TOKEN"] = _hf
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf
else:
    os.environ.pop("HF_TOKEN", None)
    os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)
    print("HF_TOKEN not set - the merged model is saved to Drive (Hub push skipped)")
# Judge model: a different family than the teacher is preferred (Groq is the
# documented first choice); this gateway fallback keeps the run self-contained.
os.environ.setdefault("JUDGE_MODEL", "deepseek/deepseek-v4-flash")
print("secrets configured:", sorted(n for n in ("TEACHER_API_KEY", "JUDGE_API_KEY", "HF_TOKEN") if os.environ.get(n)))

In [ ]:
# 2. GPU check + optional Drive checkpoints
import torch

assert torch.cuda.is_available(), "Runtime > Change runtime type > select a T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB | torch {torch.__version__}")

MOUNT_DRIVE = True  # Drive is the delivery path for the merged model
if MOUNT_DRIVE:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/supply_chain_checkpoints"
    os.makedirs(DRIVE_DIR, exist_ok=True)
else:
    DRIVE_DIR = None
assert DRIVE_DIR, "Mount Google Drive (or set MOUNT_DRIVE = False and accept a local-only model)"

In [ ]:
# 3. Project bootstrap + dataset load
import sys
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "src" / "config.py").exists() else cwd / "supply_chain_finetuning"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src import config


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


import json

train = load_jsonl(config.TRAIN_JSONL)
valid = load_jsonl(config.VALID_JSONL)
test = load_jsonl(config.TEST_JSONL)
print(f"train={len(train)} valid={len(valid)} test={len(test)}")

example = train[0]
print("--- rendered example ---")
print("SYSTEM:", example["messages"][0]["content"][:160], "...")
print("USER:", json.dumps(json.loads(example["messages"][1]["content"]), indent=1)[:700])
print("ASSISTANT:", json.dumps(json.loads(example["messages"][2]["content"]), indent=1)[:700])

In [ ]:
# 4. Token-length check (measured, not assumed)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(config.STUDENT_MODEL_ID, trust_remote_code=True)
lengths = [len(tokenizer.apply_chat_template(ex["messages"], tokenize=True))
           for ex in train + valid + test]
lengths_sorted = sorted(lengths)
p95 = lengths_sorted[int(config.TOKEN_FIT_SHARE * (len(lengths_sorted) - 1))]
share = sum(1 for n in lengths if n <= config.MAX_SEQ_LENGTH) / len(lengths)
print(f"tokens: min={min(lengths)} median={lengths_sorted[len(lengths_sorted)//2]} "
      f"p95={p95} max={max(lengths)} | within {config.MAX_SEQ_LENGTH}: {share:.1%}")
assert share >= config.TOKEN_FIT_SHARE, "sequence budget too small for the dataset"

## 5. Chat-template verification and student selection

Phi-3's template must render the system turn. If it does not, the notebook
**automatically** switches to the documented fallback
(`Qwen/Qwen2.5-1.5B-Instruct`, Apache-2.0) with its own target modules, and
prints which model is in use.

In [ ]:
rendered = tokenizer.apply_chat_template(train[0]["messages"], tokenize=False)
template_ok = train[0]["messages"][0]["content"][:60] in rendered

if template_ok:
    STUDENT_ID = config.STUDENT_MODEL_ID
    TARGET_MODULES = list(config.PHI3_TARGET_MODULES)
else:
    STUDENT_ID = config.FALLBACK_STUDENT_MODEL_ID
    TARGET_MODULES = list(config.FALLBACK_TARGET_MODULES)
    print("[warn] system turn missing from the rendered template - switching to the fallback model")
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=True)
    rendered = tokenizer.apply_chat_template(train[0]["messages"], tokenize=False)
    assert train[0]["messages"][0]["content"][:60] in rendered, "fallback template also dropped the system turn"

print("student model:", STUDENT_ID)
print("target modules:", TARGET_MODULES)

## 6. Base-model baseline FIRST (before any training)

The base student is loaded in 4-bit and generates on all 15 held-out prompts
with the SAME system prompt the fine-tuned model will see, greedy decoding
(`do_sample=False`), `max_new_tokens=512`. Identical decoding for both arms is
what makes the later comparison fair.

In [ ]:
import gc

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 has no bf16
)

base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb_config, device_map="auto",
    attn_implementation="sdpa", trust_remote_code=True,
)
base_model.eval()


def generate_arm(model, tok, records, max_new_tokens=512):
    """Greedy generation on the held-out prompts (identical decoding per arm)."""
    outputs = []
    for record in records:
        messages = [
            {"role": "system", "content": record["messages"][0]["content"]},
            {"role": "user", "content": record["messages"][1]["content"]},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
        text = tok.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        outputs.append({
            "tuple_id": record["meta"]["tuple_id"],
            "input_text": record["messages"][1]["content"],
            "raw_output": text.strip(),
        })
    gc.collect()
    torch.cuda.empty_cache()
    return outputs


base_generations = generate_arm(base_model, tokenizer, test)
config.ensure_dirs()
with open(config.BASE_GENERATIONS_JSON, "w", encoding="utf-8") as handle:
    json.dump(base_generations, handle, indent=2, ensure_ascii=False)
print(f"base generations saved: {len(base_generations)} -> {config.BASE_GENERATIONS_JSON}")
print("sample raw output:\n" + base_generations[0]["raw_output"][:400])

del base_model  # free GPU memory before training
gc.collect()
torch.cuda.empty_cache()
print("base model unloaded")

## 7. QLoRA hyperparameters (each with a written reason — also in TRAINING_NOTES.md)

| Parameter | Value | Reason |
|---|---|---|
| Quantization | NF4 + double quant, fp16 compute | best quality/size trade (QLoRA paper); T4 has no bf16 |
| LoRA r / alpha | 16 / 32 | r=8 underfits multi-sentence reasoning at ~120 examples; alpha/r=2 standard scaling |
| target_modules | `qkv_proj, o_proj, gate_up_proj, down_proj` | Phi-3 fuses Q/K/V and gate/up — split names do not exist in this architecture; all-linear per the QLoRA paper |
| lora_dropout | 0.05 | light regularization without destroying signal |
| learning rate | 2e-4, cosine, warmup 0.05 | standard QLoRA range; smooth decay for few steps |
| epochs | 3 | enough to show a validation-loss trend |
| batch 2 x accum 4 | effective 8 | T4-safe; ~15 steps/epoch (~45 total) for visible per-epoch curves |
| max seq length | 1024 | measured: >= 95% of examples fit |
| optimizer | paged_adamw_32bit | absorbs VRAM spikes |
| precision / checkpointing | fp16, grad ckpt on, use_cache off | T4 memory budget |
| attention | sdpa | FlashAttention-2 unsupported on T4 |
| seed | 42 | reproducibility |
| loss scope | assistant tokens only | matches the deployment objective |

In [ ]:
# 8. QLoRA setup (4-bit reload, k-bit preparation, LoRA adapters)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb_config, device_map="auto",
    attn_implementation="sdpa", trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

In [ ]:
# 9. Dataset for TRL (prompt/completion layout) + version-robust SFTConfig
import inspect

from datasets import Dataset
from trl import SFTConfig, SFTTrainer


def to_prompt_completion(example):
    messages = example["messages"]
    return {"prompt": messages[:-1], "completion": messages[-1]}


train_ds = Dataset.from_list([to_prompt_completion(ex) for ex in train])
valid_ds = Dataset.from_list([to_prompt_completion(ex) for ex in valid])

sft_fields = set(inspect.signature(SFTConfig.__init__).parameters)
sft_kwargs = dict(
    output_dir="outputs/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=0.3,
    optim="paged_adamw_32bit",
    fp16=True,
    gradient_checkpointing=True,
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
)
# TRL renamed the sequence-length argument between releases:
for key in ("max_seq_length", "max_length"):
    if key in sft_fields:
        sft_kwargs[key] = config.MAX_SEQ_LENGTH
        print(f"sequence-length argument: {key}")
        break
# Assistant-token-only loss: newer TRL exposes a config flag; the
# prompt/completion dataset layout restricts the loss for the others.
if "completion_only_loss" in sft_fields:
    sft_kwargs["completion_only_loss"] = True
    print("loss scope: completion_only_loss=True")
else:
    print("loss scope: prompt/completion dataset layout (prompt tokens masked)")

trainer_kwargs = dict(model=model, args=SFTConfig(**sft_kwargs),
                      train_dataset=train_ds, eval_dataset=valid_ds)
trainer_sig = set(inspect.signature(SFTTrainer.__init__).parameters)
trainer_kwargs["processing_class" if "processing_class" in trainer_sig else "tokenizer"] = tokenizer
trainer = SFTTrainer(**trainer_kwargs)
print("trainer ready")

In [ ]:
# 10. Training with the OOM playbook attached
def train_with_oom_playbook(trainer_obj):
    """On CUDA OOM: print the verbatim error and the escalation order."""
    try:
        return trainer_obj.train()
    except torch.cuda.OutOfMemoryError as exc:
        print("CUDA OOM - verbatim error:\n" + str(exc))
        print(
            "Escalation order (TRAINING_NOTES.md):\n"
            "  1. max sequence length 1024 -> 768\n"
            "  2. per-device batch 2 -> 1 (accumulation 4 -> 8)\n"
            "  3. drop MLP targets (gate_up_proj, down_proj)\n"
            "  4. restart the runtime, then retry\n"
            "  5. LoRA r 16 -> 8 (last resort)"
        )
        raise


train_result = train_with_oom_playbook(trainer)
if DRIVE_DIR:
    trainer.save_model(DRIVE_DIR + "/last_checkpoint")
    print("checkpoint mirrored to Drive")

from transformers.trainer_utils import get_last_checkpoint

LAST_CHECKPOINT = get_last_checkpoint(str(config.OUTPUTS_DIR / "checkpoints"))
print("last checkpoint:", LAST_CHECKPOINT)

In [ ]:
# 11. Loss monitoring: per-epoch table, curves, and the honest verdict
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_rows = [(round(h["epoch"], 2), h["loss"]) for h in history if "loss" in h]
eval_rows = [(round(h["epoch"], 2), h["eval_loss"]) for h in history if "eval_loss" in h]
print("epoch | train loss | validation loss")
for epoch, _ in eval_rows:
    train_at = next((loss for e, loss in train_rows if abs(e - epoch) < 0.01), None)
    val_at = next((loss for e, loss in eval_rows if abs(e - epoch) < 0.01), None)
    print(f"  {epoch}  |   {train_at}   |  {val_at}")

fig, ax = plt.subplots(figsize=(7, 4))
if train_rows:
    ax.plot(*zip(*train_rows), marker="o", label="train loss")
if eval_rows:
    ax.plot(*zip(*eval_rows), marker="s", label="validation loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

val_losses = [loss for _, loss in eval_rows]
decreased = all(val_losses[i + 1] < val_losses[i] for i in range(len(val_losses) - 1))
print("validation loss decreased at every epoch:", decreased)
if not decreased and len(val_losses) > 1:
    print(
        "pre-committed response (TRAINING_NOTES.md): rerun at learning rate 1e-4 "
        "or for 2 epochs; if the train/val gap is large, raise dropout to 0.1. "
        "Record every rerun truthfully."
    )

## 12. Merge, model card, save to Drive (Hub push optional)

The base is reloaded in **fp16** (never merge into a 4-bit base), the adapters
are attached and `merge_and_unload()` produces a standalone model. A Drive zip
is the documented alternate delivery path.

In [ ]:
from peft import PeftModel

HF_REPO_ID = os.environ.get("HF_USER", "")  # optional: "<user>/supply-chain-anomaly-analyst"

adapters_dir = str(LAST_CHECKPOINT) if LAST_CHECKPOINT else trainer.args.output_dir
base_fp16 = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, torch_dtype=torch.float16, trust_remote_code=True, device_map="auto",
)
merged = PeftModel.from_pretrained(base_fp16, adapters_dir)
merged = merged.merge_and_unload()
merged_dir = "outputs/merged_model"
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)

card = (
    "---\n"
    f"base_model: {STUDENT_ID}\n"
    "tags: [supply-chain, qlora, lora, phi-3]\n"
    "---\n"
    f"# Supply-Chain Anomaly Analyst\n\n"
    f"Fine-tuned from {STUDENT_ID} with QLoRA (4-bit NF4, r=16, alpha=32) on "
    f"{len(train)} teacher-generated event feeds (120 train / 15 valid; taxonomy "
    "and JSON output contract in the system prompt). Loss on assistant tokens "
    f"only. Final train loss {train_rows[-1][1] if train_rows else 'n/a'}, "
    f"validation loss {eval_rows[-1][1] if eval_rows else 'n/a'}.\n"
)
with open(merged_dir + "/README.md", "w", encoding="utf-8") as handle:
    handle.write(card)

import shutil

archive = shutil.make_archive(DRIVE_DIR + "/merged_model", "zip", merged_dir)
print("merged model saved to Drive:", archive)
print("(set the Drive folder to 'anyone with the link' to satisfy the"
      " model-link requirement)")

if HF_REPO_ID and os.environ.get("HF_TOKEN"):
    merged.push_to_hub(HF_REPO_ID, private=False)
    tokenizer.push_to_hub(HF_REPO_ID)
    print("pushed to:", "https://huggingface.co/" + HF_REPO_ID)
else:
    print("Hub push skipped (no HF_TOKEN / HF_USER)")

In [ ]:
# 13. Tuned generations: SAME prompts, SAME decoding as the base arm
tuned_generations = generate_arm(merged, tokenizer, test)
with open(config.TUNED_GENERATIONS_JSON, "w", encoding="utf-8") as handle:
    json.dump(tuned_generations, handle, indent=2, ensure_ascii=False)
print(f"tuned generations saved: {len(tuned_generations)} -> {config.TUNED_GENERATIONS_JSON}")
print("sample raw output:\n" + tuned_generations[0]["raw_output"][:400])

## 14. Evaluation — ROUGE-L, BERTScore, programmatic checks

Both arms are scored on the identical 15 held-out prompts after canonicalizing
their outputs (sorted keys, rounded floats). Unparseable outputs stay in the
comparison and count as schema failures.

In [ ]:
import pandas as pd

from evaluation.metrics import evaluate_arm
from evaluation.normalize import normalize_output

records_by_id = {
    record["meta"]["tuple_id"]: {
        "input_feed": json.loads(record["messages"][1]["content"]),
        "gold_assessment": json.loads(record["messages"][2]["content"]),
    }
    for record in test
}
base_report = evaluate_arm(base_generations, records_by_id, normalize_output, include_bertscore=True)
tuned_report = evaluate_arm(tuned_generations, records_by_id, normalize_output, include_bertscore=True)

summary_rows = [
    {"metric": "ROUGE-L (canonical JSON)", "base": base_report["rouge_l"]["mean"], "fine-tuned": tuned_report["rouge_l"]["mean"]},
    {"metric": "ROUGE-L (root cause)", "base": base_report["rouge_l_root_cause"]["mean"], "fine-tuned": tuned_report["rouge_l_root_cause"]["mean"]},
    {"metric": "BERTScore F1", "base": base_report["bertscore_f1"]["mean"], "fine-tuned": tuned_report["bertscore_f1"]["mean"]},
]
for check in base_report["checks"]:
    summary_rows.append({"metric": check, "base": base_report["checks"][check],
                         "fine-tuned": tuned_report["checks"][check]})
summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))
with open(config.METRICS_SUMMARY_JSON, "w", encoding="utf-8") as handle:
    json.dump({"base": {k: v for k, v in base_report.items() if k != "rows"},
               "tuned": {k: v for k, v in tuned_report.items() if k != "rows"}},
              handle, indent=2, default=str)

## 15. Blinded LLM-as-judge

The judge scores the INPUT and one output at a time; it never sees the gold
answer. Both arms are shuffled into anonymous slots; a consistency gauge
re-scores three outputs (target: per-dimension differences within ±1).

In [ ]:
from evaluation.judge import (
    blind, consistency_gauge, judge_item, unblind, warn_if_judge_is_teacher,
)
from src.llm_client import LLMClient, settings_from_env

warn_if_judge_is_teacher()  # visible warning if judge == teacher (self-preference risk)

judge_client = LLMClient(
    settings=settings_from_env(
        "JUDGE", config.DEFAULT_TEACHER_BASE_URL,
        os.environ.get(config.ENV_JUDGE_MODEL, ""), config.JUDGE_TEMPERATURE,
    ),
    cache_path=config.JUDGE_CACHE_JSON,
    usage_log_path=config.JUDGE_USAGE_CSV,
    repair_attempts=1,
)

items = [
    {"tuple_id": gen["tuple_id"], "arm": arm,
     "input_feed": records_by_id[gen["tuple_id"]]["input_feed"],
     "raw_output": gen["raw_output"]}
    for arm, generations in (("base", base_generations), ("tuned", tuned_generations))
    for gen in generations
]
blinded, truths = blind(items, seed=config.SEED)
scored = []
for item in blinded:
    result = judge_item(judge_client, item["input_feed"], item["raw_output"], salt=str(item["slot"]))
    scored.append({**item, "result": result})
report = unblind(scored, truths)
print(pd.DataFrame([
    {"arm": arm, "mean total (0-30)": values["mean_total"],
     **{dim: values["per_dimension"][dim] for dim in values["per_dimension"]}}
    for arm, values in report["summary"].items()
]).to_string(index=False))

gauge = consistency_gauge(judge_client, blinded)
print("consistency gauge (slot, max dimension diff, within ±1):",
      [(g["slot"], g["max_diff"], g["within_tolerance"]) for g in gauge])

with open(config.JUDGE_RESULTS_JSON, "w", encoding="utf-8") as handle:
    json.dump({"summary": report["summary"], "gauge": gauge}, handle, indent=2, default=str)

In [ ]:
# 16. Manual hallucination audit: template now, human labels next
from evaluation.manual_audit import compute_rate, make_template

make_template(tuned_generations, records_by_id, config.AUDIT_TEMPLATE_CSV)
print("audit template written:", config.AUDIT_TEMPLATE_CSV)
compute_rate(config.AUDIT_TEMPLATE_CSV)  # prints "manual labels pending" until labelled

## Combined results

*(produced from the real runs above — nothing pre-filled)*

In [ ]:
combined = pd.DataFrame([
    {"metric": "ROUGE-L (canonical JSON)", "base": base_report["rouge_l"]["mean"], "fine-tuned": tuned_report["rouge_l"]["mean"]},
    {"metric": "ROUGE-L (root cause)", "base": base_report["rouge_l_root_cause"]["mean"], "fine-tuned": tuned_report["rouge_l_root_cause"]["mean"]},
    {"metric": "BERTScore F1", "base": base_report["bertscore_f1"]["mean"], "fine-tuned": tuned_report["bertscore_f1"]["mean"]},
    {"metric": "Judge mean total (0-30)",
     "base": report["summary"].get("base", {}).get("mean_total"),
     "fine-tuned": report["summary"].get("tuned", {}).get("mean_total")},
    {"metric": "Valid JSON %", "base": base_report["checks"]["valid_json_pct"], "fine-tuned": tuned_report["checks"]["valid_json_pct"]},
    {"metric": "Class exact-match %", "base": base_report["checks"]["class_exact_pct"], "fine-tuned": tuned_report["checks"]["class_exact_pct"]},
    {"metric": "Severity within ±1 %", "base": base_report["checks"]["severity_within_1_pct"], "fine-tuned": tuned_report["checks"]["severity_within_1_pct"]},
    {"metric": "Evidence-subset %", "base": base_report["checks"]["evidence_subset_pct"], "fine-tuned": tuned_report["checks"]["evidence_subset_pct"]},
    {"metric": "Ground-guard violations", "base": base_report["checks"]["ground_guard_violation_count"], "fine-tuned": tuned_report["checks"]["ground_guard_violation_count"]},
])
print(combined.to_string(index=False))

## Qualitative analysis

*(to be written after inspecting the real outputs above — two paragraphs:*

1. *where fine-tuning helped, with two real before/after excerpts from the
   generations and the metric deltas;*
2. *remaining failure modes (e.g. rare families, severity calibration) and the
   next steps that would address them — more data in weak families, a quality
   filter on teacher output, DPO for severity.)*

## Appendix: teacher system prompt (verbatim; also in data/teacher_system_prompt.txt)

```
You are a senior supply-chain data engineer generating ONE synthetic
warehouse/SKU-day event feed and its expert assessment.

Return ONLY a JSON object with exactly these two top-level keys:
{
  "input_feed": {
    "product_category": "<category given>",
    "region": "<region given>",
    "order_volume_vs_forecast_pct": <number, % vs forecast>,
    "inventory_days_on_hand": <number, days>,
    "on_time_delivery_pct": <number, %>,
    "supplier_status": "<short status, e.g. normal | delayed | partial | halted>",
    "transit_days_normal": <number, days>,
    "transit_days_observed": <number, days>,
    "port_congestion_index": <number, 0-100>,
    "weather_event": "<none | the specific event>",
    "labor_document_note": "<none | the specific labour note>",
    "unit_cost_vs_last_quarter_pct": <number, % change>,
    "notes": "<one plausible free-text sentence consistent with the story>"
  },
  "gold_assessment": {
    "is_anomaly": <true|false>,
    "anomaly_class": "<one class from the taxonomy>",
    "severity": <integer 1-5 within the allowed band>,
    "root_cause": "<2-4 sentences grounded ONLY in the feed>",
    "evidence_fields": ["<input field names that justify the class>"],
    "corrective_actions": ["<3-5 concrete actions>"],
    "confidence": <number 0-1>
  }
}

Grounding rules:
- The numbers MUST agree with the story. Example: a port congestion case has a high
  port_congestion_index AND transit_days_observed > transit_days_normal; unrelated
  fields stay near their normal ranges.
- evidence_fields must contain ONLY input field names that actually exist above, and
  must be non-empty when is_anomaly is true.
- The root cause must NOT mention any event (storm, strike, outage, ...) that the feed
  does not explicitly contain.
- For a normal case: is_anomaly=false, anomaly_class="none", severity=1, and the feed
  shows only benign fluctuations with all operational metrics near normal.
- Taxonomy classes: demand_spike, demand_collapse, supplier_shipment_delay,
  port_logistics_congestion, weather_disruption, quality_hold, labor_strike,
  freight_inflation, inventory_obsolescence, warehouse_capacity_breach, none.

Forbidden: any text outside the JSON object, extra top-level keys, events or fields
not present in the feed, severity outside the allowed band.
```

## Appendix: student system prompt (SFT system turn AND base baseline)

```
You are a supply-chain anomaly analyst. You receive one event feed as a
JSON object and must return a single JSON assessment object and nothing else.

Output schema (all fields required):
{
  "is_anomaly": true | false,
  "anomaly_class": "demand_spike" | "demand_collapse" | "supplier_shipment_delay" |
                   "port_logistics_congestion" | "weather_disruption" | "quality_hold" |
                   "labor_strike" | "freight_inflation" | "inventory_obsolescence" |
                   "warehouse_capacity_breach" | "none",
  "severity": <integer 1 (trivial) to 5 (critical)>,
  "root_cause": "<2-4 sentences grounded only in the feed>",
  "evidence_fields": ["<input field names that justify the class>"],
  "corrective_actions": ["<3-5 concrete, actionable steps>"],
  "confidence": <number between 0 and 1>
}

Rules:
- Use anomaly_class "none" with is_anomaly false for normal feeds.
- Cite only input field names that exist in the feed.
- Never mention events that the feed does not contain.
- Severity guide: 1 trivial, 2 minor, 3 material, 4 severe, 5 critical.

Return ONLY the JSON object.
```